# C3: dados ? treinamento ? bin?rio ? API por modelo
Este notebook usa o SDK do MLflow e o SDK local `src.workbench`. N?o abre conex?o SQL: metadados seguem por servi?os para PostgreSQL; bin?rios seguem para RustFS.
Base de teste: 350 observa??es sint?ticas, 30 s?ries semestrais, lacuna e curso novo. Nenhum dado pessoal. Comece com uma s?rie; altere `MODEL_KEY=None` para treinar as 30.
Execute ap?s reconstruir as imagens e iniciar o servi?o `ingestion`. Cada execu??o de treinamento cria uma nova vers?o, sem aprova??o autom?tica.

In [ ]:
from src.workbench.sdk import connect, submit_dataset, test_dataset, train
connect()

## Receber e validar os dados no servi?o dedicado
O recebimento ? idempotente pelo conte?do. Rejeita campos extras, negativos, duplicidades, refer?ncias inconsistentes e vazamento temporal.

In [ ]:
payload = test_dataset()
snapshot = submit_dataset(payload)
{key: snapshot[key] for key in ('id', 'cycle_id', 'row_count', 'sha256', 'source_reference')}

## Treinar e persistir
O treinamento salva um modelo ajustado, l? todos os arquivos de volta do RustFS, valida SHA256 e compara a previs?o antes de registrar a vers?o.

In [ ]:
MODEL_KEY = "C1-CR1-P1"
versions = train(snapshot['id'], model_key=MODEL_KEY)
versions

## Aprova??o e publica??o expl?citas
No terminal da raiz, use a vers?o retornada acima (n?o presuma vers?o 1):
```powershell
docker compose run --rm tools promote C1-CR1-P1 VERSAO operador "validacao da POC"
.\deployment\scripts\compose.ps1 publish -ModelKey C1-CR1-P1
```
A publica??o cria um container/API exclusivo para o modelo. A porta local ? exibida pelo script. Uma nova aprova??o exige republicar esse servi?o. O servi?o antigo recusa a vers?o revogada.
N?o execute a pr?xima c?lula antes de concluir essa etapa.

In [ ]:
import os, httpx
response = httpx.post(os.environ['API_URI'] + '/v1/inference/predict',
    headers={'Authorization': 'Bearer ' + os.environ['API_TOKEN']},
    json={'cycle_id': snapshot['cycle_id'], 'snapshot_id': snapshot['id'],
          'model_key': MODEL_KEY, 'requested_by': 'notebook-c3'}, timeout=120)
response.raise_for_status()
response.json()